### Graph V: Looping Graph - Practice & Exercises
Welcome to the fifth LangGraph notebook! In this notebook, you will practice implementing iterative/looping logic in your AI agent workflows using LangGraph.

#### Part 1: Follow-Along Practice
##### GoalWe want to build a looping graph that routes execution back to a previous node until a specific condition is met:
```
        ┌───────────┐
        │   START   │
        └─────┬─────┘
              │
              ▼
        ┌───────────┐
   ┌───►│   guess   │
   │    └─────┬─────┘
   │          │
   │          ▼
   │    ┌───────────┐
   │    │ evaluate  │
   │    └─────┬─────┘
   │          │
   │          ▼
   │    /───────────\
   │   <  Is correct >
   │    \───────────/
   │      /       \
   │    No         Yes
   └─────/         \───────► [ END ]
```

##### Steps:1. Define the `AgentState` schema tracking the target number, guess, attempts, feedback, and won flag.
2. Create the node functions (`guess_node`, `evaluate_node`) and the routing function (`should_continue`).
3. Build the graph with a conditional loop connection.
4. Compile and invoke the graph.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. Define the State Schema
class AgentState(TypedDict):
    target: int
    guess: int
    attempts: int
    feedback: str
    won: bool

In [ ]:
# 2. Define Nodes and Routing Logic
import random

def guess_node(state: AgentState) -> AgentState:
    state["guess"] = random.randint(1, 10)
    state["attempts"] = state.get("attempts", 0) + 1
    return state

def evaluate_node(state: AgentState) -> AgentState:
    guess = state["guess"]
    target = state["target"]
    if guess == target:
        state["feedback"] = "Correct!"
        state["won"] = True
    elif guess < target:
        state["feedback"] = "Higher"
        state["won"] = False
    else:
        state["feedback"] = "Lower"
        state["won"] = False
    return state

def should_continue(state: AgentState) -> str:
    if state["won"] or state["attempts"] >= 5:
        return "stop"
    else:
        return "go_on"

In [ ]:
# 3. Build and Compile the Graph
graph = StateGraph(AgentState)

graph.add_node("guesser", guess_node)
graph.add_node("evaluator", evaluate_node)

graph.add_edge(START, "guesser")
graph.add_edge("guesser", "evaluator")

graph.add_conditional_edges(
    "evaluator",
    should_continue,
    {
        "go_on": "guesser",
        "stop": END
    }
)

app = graph.compile()

In [ ]:
# 4. Invoke the Graph
result = app.invoke({"target": 7, "attempts": 0, "won": False})
print(f"Target: {result['target']}, Final Guess: {result['guess']}, Attempts: {result['attempts']}, Won: {result['won']}")

---

#### Part 2: Practice Exercises
##### Exercise 1: Automatic Higher or Lower Game
###### GoalImplement an **Automatic Higher or Lower Game** using binary search logic, looping the guesser and evaluator nodes until the target number is guessed successfully (within a maximum of 7 attempts).

###### Sample Input```python
{
    "target": 13,
    "lower_bound": 1,
    "upper_bound": 20,
    "attempts": 0,
    "won": False
}
```

###### Sample Output```python
"Attempts: 3, Last Guess: 13, Won: True"
```

###### Plan1. **Define State Schema:** Create `AgentState` containing `target` (int), `guess` (int), `lower_bound` (int), `upper_bound` (int), `attempts` (int), `won` (bool), and `feedback` (str).
2. **Define Node Functions:** Write `guesser_node` to compute the binary search midpoint `(lower_bound + upper_bound) // 2` as the guess, and `evaluator_node` to check the guess and adjust boundaries (`lower_bound = guess + 1` if lower, or `upper_bound = guess - 1` if higher).
3. **Define Routing Logic:** Write a function `check_loop` to decide whether to stop (`won == True` or `attempts >= 7`) or continue looping.
4. **Build, Compile & Invoke Graph:** Initialize the `StateGraph`, hook up nodes, specify conditional routing, compile the graph, invoke it with the sample input, and print the results.

In [ ]:
# TODO: Define your State, Nodes, Routing Logic, Graph, Compile, and Invoke it here


<details>
<summary>💡 Click to see solution</summary>

```python
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

### 1. State Schemaclass AgentState(TypedDict):
    target: int
    guess: int
    lower_bound: int
    upper_bound: int
    attempts: int
    won: bool
    feedback: str

### 2. Node Functionsdef guesser_node(state: AgentState) -> AgentState:
    state["guess"] = (state["lower_bound"] + state["upper_bound"]) // 2
    state["attempts"] = state.get("attempts", 0) + 1
    return state

def evaluator_node(state: AgentState) -> AgentState:
    guess = state["guess"]
    target = state["target"]
    if guess == target:
        state["won"] = True
        state["feedback"] = "Correct"
    elif guess < target:
        state["lower_bound"] = guess + 1
        state["won"] = False
        state["feedback"] = "Higher"
    else:
        state["upper_bound"] = guess - 1
        state["won"] = False
        state["feedback"] = "Lower"
    return state

### Routing Logicdef check_loop(state: AgentState) -> str:
    if state["won"] or state["attempts"] >= 7:
        return "end"
    return "loop"

### 3. Create Graphgraph = StateGraph(AgentState)
graph.add_node("guesser", guesser_node)
graph.add_node("evaluator", evaluator_node)

graph.add_edge(START, "guesser")
graph.add_edge("guesser", "evaluator")

graph.add_conditional_edges(
    "evaluator",
    check_loop,
    {
        "loop": "guesser",
        "end": END
    }
)

### 4. Compile and Runapp = graph.compile()
output = app.invoke({
    "target": 13,
    "lower_bound": 1,
    "upper_bound": 20,
    "attempts": 0,
    "won": False
})
print(f"Attempts: {output['attempts']}, Last Guess: {output['guess']}, Won: {output['won']}")
```
</details>